# 2. Do the two backends compute the same thing?

Speed is worth something only if the results are right. This notebook computes the same seismograms and the
same gradient on the four engines (two backends × two numeric modes) and compares them. Two kinds of
difference show up, of very different sizes:

* **between the backends, in the same numeric mode**: the same mathematics, compiled by two different
  compilers, which round a few operations in another order. A computer keeps about 7 significant digits of
  every number here (single precision), so the results agree to about 6 digits;
* **between the numeric modes**: `legacy` keeps the bugs of PyFWI 0.1.10, `fixed` corrects them. That is a
  real change of the mathematics, and the results differ by percents, sometimes tens of percents.

It takes less than a minute.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from comparison import (  # the code shared by the notebooks of this folder: comparison.py
    CUDA_FIXED,
    CUDA_LEGACY,
    ENGINES,
    OPENCL_FIXED,
    adjoint_source,
    check_engines,
    colour,
    inclusion_model,
    line_style,
    name,
    propagator,
    relative_difference,
    style,
    survey,
)

check_engines()

## Settings

The example of the PyFWI documentation: a 100 × 100 model with 7 m cells, 4 sources, 0.6 s of recording.
`SHOT` and `RECEIVER` choose what the figures show.

In [ ]:
NZ, NX, NS, T = 100, 100, 4, 0.6
SHOT = 1  # 0 to NS - 1
RECEIVER = 60  # a receiver of the shot, 0 to 95

## Running the four engines

On every engine: the seismograms of the true model (the "observed" data), then one iteration of an inversion
from a smoothed model: its seismograms, the misfit, and the gradient of the misfit with respect to vp, vs and
the density. Each engine runs twice, to check that a backend gives the same bits from one run to the next.

In [ ]:
problem = survey(NZ, NX, NS, T)
true, start = inclusion_model(NZ, NX), inclusion_model(NZ, NX, smooth=1)


def run(engine):
    W = propagator(problem, engine)
    observed = W.forward_modeling(true, show=False)
    predicted = W.forward_modeling(start, show=False)
    misfit, source = adjoint_source(W, predicted, observed)
    return {"seismograms": observed, "misfit": misfit, "gradient": W.gradient(source, show=False)}


results, repeated = {}, {}
for engine in ENGINES:
    results[engine], repeated[engine] = run(engine), run(engine)
    print(f"{name(engine):15s} misfit {results[engine]['misfit']:.6e}   "
          f"second run identical: {relative_difference(repeated[engine]['seismograms'], results[engine]['seismograms']) == 0 and relative_difference(repeated[engine]['gradient'], results[engine]['gradient']) == 0}")

## How far apart the results are

The relative difference ‖a − b‖ / ‖b‖ over all the numbers of a result: 1e-6 means the results agree to about
six digits, 1e-2 that they differ by about 1 %.

In [ ]:
for what in ("seismograms", "gradient"):
    print(f"\n{what}: relative difference of the row engine from the column engine")
    print(f"{'':16s}" + "".join(f"{name(engine):>16s}" for engine in ENGINES))
    for a in ENGINES:
        print(f"{name(a):16s}" + "".join(f"{relative_difference(results[a][what], results[b][what]):16.1e}"
                                        for b in ENGINES))

Read the table by pairs:

* **OpenCL, fixed against CUDA, fixed** and **OpenCL, legacy against CUDA, legacy**: the same mathematics on
  the two backends. The results differ by a few millionths (a few e-06): the rounding of single-precision
  numbers. The gradients a little more than the seismograms, because a gradient adds up many small
  contributions, each rounded.
* **legacy against fixed**, on either backend: the corrections of the `fixed` mode, a real difference, of the
  order of e-01, tens of percents. On the seismograms it sits mostly at the receivers next to each source,
  where the `legacy` mode adds the source once every few internal time steps instead of at every step (the
  figures below show it); on the gradients it is everywhere (the gradients of vs and of the density of PyFWI
  0.1.10 were wrong; the [user guide](../../docs/cuda-migration/user-guide.md) lists every correction).

## Seeing the differences

The seismograms of one shot (time downwards, one column per receiver), and the two kinds of difference on the
same colour scale. The difference between the backends is invisible at this scale; the last panel enlarges it
100 000 times.

In [ ]:
receivers = results[CUDA_FIXED]["seismograms"]["taux"].shape[1] // NS
shot = slice(SHOT * receivers, (SHOT + 1) * receivers)


def section(engine):
    return results[engine]["seismograms"]["taux"][:, shot]


clip = 0.2 * np.abs(section(CUDA_FIXED)).max()
panels = {
    "CUDA, fixed": section(CUDA_FIXED),
    "CUDA − OpenCL\n(both fixed)": section(CUDA_FIXED) - section(OPENCL_FIXED),
    "fixed − legacy\n(both CUDA)": section(CUDA_FIXED) - section(CUDA_LEGACY),
    "CUDA − OpenCL (both fixed)\n× 100 000": 1e5 * (section(CUDA_FIXED) - section(OPENCL_FIXED)),
}
fig, axes = plt.subplots(1, 4, figsize=(14, 4.5), sharey=True)
for ax, (title, data) in zip(axes, panels.items()):
    ax.imshow(data, cmap="gray", vmin=-clip, vmax=clip, aspect="auto", extent=[0, receivers, T, 0])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("receiver")
axes[0].set_ylabel("time (s)")
fig.suptitle(f"Shot {SHOT}: seismograms and differences, same grey scale", x=0.01, ha="left")
fig.tight_layout()
plt.show()

One trace, the recording of one receiver: the four engines on top (the two `fixed` curves lie on each other,
and so do the two `legacy` ones), and below the size of the two differences, on a logarithmic scale, relative to
the largest amplitude of the trace.

In [ ]:
trace = {engine: section(engine)[:, RECEIVER] for engine in ENGINES}
time_axis = np.linspace(0, T, len(trace[CUDA_FIXED]))
peak = np.abs(trace[CUDA_FIXED]).max()

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
style(top)
for engine in ENGINES:
    top.plot(time_axis, trace[engine], line_style(engine), color=colour(engine), linewidth=1.5, label=name(engine))
top.legend(frameon=False, fontsize=9, ncol=4)
top.set_title(f"Shot {SHOT}, receiver {RECEIVER}", loc="left", fontsize=10)

style(bottom)
tiny = 1e-12 * peak  # where two numbers are equal, the logarithmic scale needs a floor
bottom.semilogy(time_axis, np.abs(trace[CUDA_FIXED] - trace[OPENCL_FIXED]) / peak + tiny / peak, color=colour(CUDA_FIXED),
                label="CUDA − OpenCL, both fixed (rounding)")
bottom.semilogy(time_axis, np.abs(trace[CUDA_FIXED] - trace[CUDA_LEGACY]) / peak + tiny / peak, color="black",
                label="fixed − legacy, both CUDA (the corrections)")
bottom.set_ylim(1e-9, 1)
bottom.set_ylabel("|difference| / largest amplitude", fontsize=9)
bottom.set_xlabel("time (s)")
bottom.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## The gradients

The gradient of the misfit with respect to vp, vs and the density: where the inversion would change the model.
Each row is one parameter, divided by its largest value. The two backends draw the same picture in the `fixed`
mode: their difference, enlarged 100 000 times, is the noise of rounding. The difference between the modes, not
enlarged, is part of the picture.

In [ ]:
PARAMETERS = {"vp": "vp", "vs": "vs", "rho": "density"}


def gradient(engine, key):
    return results[engine]["gradient"][key]


for key, text in PARAMETERS.items():
    reference = np.linalg.norm(gradient(CUDA_FIXED, key))
    backend = np.linalg.norm(gradient(CUDA_FIXED, key) - gradient(OPENCL_FIXED, key)) / reference
    mode = np.linalg.norm(gradient(CUDA_FIXED, key) - gradient(CUDA_LEGACY, key)) / reference
    print(f"gradient of {text:8s} CUDA - OpenCL (both fixed) {backend:.1e}   fixed - legacy (both CUDA) {mode:.1e}")

fig, axes = plt.subplots(3, 3, figsize=(12, 10), sharex=True, sharey=True)
for row, (key, text) in zip(axes, PARAMETERS.items()):
    scale = np.abs(gradient(CUDA_FIXED, key)).max()
    panels = {
        f"{text}: CUDA, fixed": gradient(CUDA_FIXED, key),
        f"{text}: CUDA − OpenCL (both fixed)\n× 100 000": 1e5 * (gradient(CUDA_FIXED, key) - gradient(OPENCL_FIXED, key)),
        f"{text}: fixed − legacy (both CUDA)": gradient(CUDA_FIXED, key) - gradient(CUDA_LEGACY, key),
    }
    for ax, (title, data) in zip(row, panels.items()):
        image = ax.imshow(data / scale, cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_title(title, fontsize=10)
    row[0].set_ylabel("depth (grid points)")
for ax in axes[-1]:
    ax.set_xlabel("x (grid points)")
fig.colorbar(image, ax=axes, shrink=0.6, label="gradient / largest value of the CUDA, fixed gradient")
plt.show()

## What this says

* **In the same numeric mode, the backend changes only the rounding**: about one millionth on the seismograms,
  a little more on the gradients. Your results do not depend on the backend.
* **The numeric mode is the real difference.** OpenCL starts in `legacy`, so that a plain installation gives the
  results of PyFWI 0.1.10; CUDA starts in `fixed`. To compare the two backends on the same footing, set the mode:
  `inpa["numerics"] = "fixed"` (or `PYFWI_NUMERICS=fixed`).
* **Each backend gives the same bits from one run to the next** (the "second run identical" above).
* A long inversion repeats these calculations many times, and tiny differences grow from iteration to iteration,
  as they would on another computer. [Notebook 3](3_inversion.ipynb) runs a complete inversion on both backends.